# Importing Necessary Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

plt.style.use('fivethirtyeight')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

# Loading The Data

In [ ]:
df = pd.read_csv('/kaggle/input/thyroid-disease-data/Thyroid_Diff.csv')

# EDA

In [ ]:
#first 5 rows
df.head()

In [ ]:
#shape of the data
df.shape

In [ ]:
#statistical information about the data
df.describe().T

In [ ]:
#number of nulls in each column
df.isnull().sum()

In [ ]:
#information about the data
df.info()

In [ ]:
#unique values in each column
for column in df.columns:
    unique_values = df[column].unique()
    print(f"Unique values in column '{column}':")
    print(unique_values)
    print()

In [ ]:
#Age distribution
plt.figure(figsize = (8, 6))
sns.histplot(df['Age'], bins = 20, kde = True, color = 'skyblue', edgecolor = 'black')
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

In [ ]:
#Gender
plt.figure(figsize = (8, 6))
sns.countplot(x = 'Gender', data = df)
plt.title('Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.show()

In [ ]:
#Smoking status
plt.figure(figsize = (8, 6))
sns.countplot(x = 'Smoking', data = df)
plt.title('Smoking Status')
plt.xlabel('Smoking')
plt.ylabel('Count')
plt.show()

In [ ]:
#History of Smoking
plt.figure(figsize = (8, 6))
sns.countplot(x = 'Hx Smoking', data = df)
plt.title('History of Smoking')
plt.xlabel('History of Smoking')
plt.ylabel('Count')
plt.show()

In [ ]:
#History of Radiotherapy
plt.figure(figsize = (8, 6))
sns.countplot(x = 'Hx Radiothreapy', data = df)
plt.title('History of Radiotherapy')
plt.xlabel('History of Radiotherapy')
plt.ylabel('Count')
plt.show()

In [ ]:
#Thyroid Function
plt.figure(figsize = (20, 10))
sns.countplot(x = 'Thyroid Function', data = df)
plt.title('Thyroid Function')
plt.xlabel('Thyroid Function')
plt.ylabel('Count')
plt.show()

In [ ]:
#Physical Examination
plt.figure(figsize = (20, 10))
sns.countplot(x = 'Physical Examination', data = df)
plt.title('Physical Examination')
plt.xlabel('Physical Examination')
plt.ylabel('Count')
plt.show()

In [ ]:
#Adenopathy
plt.figure(figsize = (8, 6))
sns.countplot(x = 'Adenopathy', data = df)
plt.title('Adenopathy')
plt.xlabel('Adenopathy')
plt.ylabel('Count')
plt.show()

In [ ]:
#Pathology
plt.figure(figsize = (8, 6))
sns.countplot(x = 'Pathology', data = df)
plt.title('Pathology')
plt.xlabel('Pathology')
plt.ylabel('Count')
plt.show()

In [ ]:
#Focality
plt.figure(figsize = (8, 6))
sns.countplot(x = 'Focality', data = df)
plt.title('Focality')
plt.xlabel('Focality')
plt.ylabel('Count')
plt.show()

In [ ]:
#Risk
plt.figure(figsize = (8, 6))
sns.countplot(x = 'Risk', data = df)
plt.title('Risk')
plt.xlabel('Risk')
plt.ylabel('Count')
plt.show()

In [ ]:
#T, N, M (TNM Staging)
plt.figure(figsize = (8, 6))
sns.countplot(x = 'T', data = df, order = sorted(df['T'].unique()))
plt.title('T Staging')
plt.xlabel('T Stage')
plt.ylabel('Count')
plt.show()

plt.figure(figsize = (8, 6))
sns.countplot(x = 'N', data = df, order = sorted(df['N'].unique()))
plt.title('N Staging')
plt.xlabel('N Stage')
plt.ylabel('Count')
plt.show()

plt.figure(figsize = (8, 6))
sns.countplot(x = 'M', data = df, order = sorted(df['M'].unique()))
plt.title('M Staging')
plt.xlabel('M Stage')
plt.ylabel('Count')
plt.show()

In [ ]:
#Stage
stage_order = ['I', 'II', 'III', 'IVA', 'IVB']

plt.figure(figsize = (8, 6))
sns.countplot(x = 'Stage', data = df, order = stage_order)  # Order the stages
plt.title('Distribution of Stage')
plt.xlabel('Stage')
plt.ylabel('Count')
plt.show()

In [ ]:
#Response
plt.figure(figsize = (15, 8))
sns.countplot(x = 'Response', data = df)
plt.title('Response')
plt.xlabel('Response')
plt.ylabel('Count')
plt.show()

In [ ]:
#Recurred
plt.figure(figsize = (8, 6))
sns.countplot(x = 'Recurred', data = df)
plt.title('Recurred')
plt.xlabel('Recurred')
plt.ylabel('Count')
plt.show()

# Data Pre-Processing

In [ ]:
#define encoding methods for each column
encoding_methods = {
    'Gender': 'label',
    'Smoking': 'label',
    'Hx Smoking': 'label',
    'Hx Radiothreapy': 'label',
    'Thyroid Function': 'one-hot',
    'Physical Examination': 'one-hot',
    'Adenopathy': 'one-hot',
    'Pathology': 'one-hot',
    'Focality': 'label',
    'Risk': 'label',
    'T': 'label',
    'N': 'label',
    'M': 'label',
    'Stage': 'label',
    'Response': 'one-hot',
    'Recurred': 'label'
}

#apply encoding to each column
for column, method in encoding_methods.items():
    if method == 'label':
        label_encoder = LabelEncoder()
        df[column] = label_encoder.fit_transform(df[column])
    elif method == 'one-hot':
        one_hot_encoder = OneHotEncoder(sparse = False, drop = 'first')
        encoded = one_hot_encoder.fit_transform(df[[column]])
        column_names = [f'{column}_{category}' for category in one_hot_encoder.categories_[0][1:]]
        df_encoded = pd.DataFrame(encoded, columns=column_names)
        df = pd.concat([df, df_encoded], axis = 1)
        df.drop(columns = [column], inplace = True)

In [ ]:
#apply Box-Cox transformation to 'Age' column
age_boxcox, _ = stats.boxcox(df['Age'])

#apply Min-Max scaling to the transformed 'Age' column
scaler = MinMaxScaler()
age_scaled = scaler.fit_transform(age_boxcox.reshape(-1, 1)).flatten()

#replace 'Age' column with the scaled values
df['Age'] = age_scaled

In [ ]:
#Age distribution after normalizing and scaling
plt.figure(figsize = (8, 6))
sns.histplot(df['Age'], bins = 20, kde = True, color = 'skyblue', edgecolor = 'black')
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

# Modelling

In [ ]:
#split data into features and target
X = df.drop(columns=['Recurred'])
y = df['Recurred']

#split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#instantiate classification models
models = {
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Logistic Regression': LogisticRegression(),
    'Support Vector Machine': SVC(),
    'Gaussian Naive Bayes': GaussianNB(),
    'AdaBoost': AdaBoostClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'XGBoost': XGBClassifier(),
    'CatBoost': CatBoostClassifier(verbose=0)
}

#fit models and generate classification reports
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred)
    print(f'{name} Classification Report:\n{report}\n')